# AI Gateway Classifier Training

This notebook trains the optional `AiClassifier` on data exported from `ai_gateway_logs`.
Expected CSV columns: `messageHash` or `message`, `decision`, `reason`, `language`, `length`.
Use raw `message` only if your export flow keeps it in a privacy-safe form.

In [ ]:
from pathlib import Path
import csv
import sys

from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

sys.path.append(str(Path("..").resolve()))

from app.services.ai_classifier import AiClassifier

DATASET_PATH = Path("../data/ai_gateway_logs.csv")
MODEL_OUTPUT_PATH = Path("../models/ai_gateway_classifier.joblib")
ON_TOPIC_DECISIONS = {"FORWARD", "LOCAL_ANSWER"}


In [ ]:
texts: list[str] = []
labels: list[int] = []

with DATASET_PATH.open("r", encoding="utf-8") as csv_file:
    reader = csv.DictReader(csv_file)
    for row in reader:
        message = (row.get("message") or "").strip()
        decision = (row.get("decision") or "").strip().upper()
        if not message or decision not in {"FORWARD", "LOCAL_ANSWER", "REJECT"}:
            continue
        texts.append(message)
        labels.append(1 if decision in ON_TOPIC_DECISIONS else 0)

len(texts), sum(labels), len(labels) - sum(labels)


In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

classifier = AiClassifier()
classifier.train(train_texts, train_labels)

predicted_labels = [1 if classifier.predict(text) >= 0.5 else 0 for text in test_texts]
print(classification_report(test_labels, predicted_labels, digits=3))

classifier.save_model(MODEL_OUTPUT_PATH)
print(f"Saved model to {MODEL_OUTPUT_PATH}")
